# Notebook 05 — The Downstream Economics of Arbitrage-Free Surfaces

NB02–NB04 built and audited three families of smoothers (SVI/SSVI, per-day derivative-constrained
deep smoother, neural operators) and established the thesis's methodological result: **soft
no-arbitrage constraints control derivatives of the surface, and can be numerically satisfied
while economically void** (the collocation blind spot; the across-days transfer gap of the
operator). This notebook closes the argument by pricing with those derivatives:

| § | Object | Why it is the constraints made visible |
|---|---|---|
| **A** | Dupire local volatility | $\sigma^2_{loc}(k,\tau) = \partial_\tau w \,/\, g(k,\tau)$ — the calendar constraint **is** the numerator, the butterfly constraint **is** the denominator. An arbitrage violation is not an abstract flag: it is a negative or exploding local variance that must be *repaired* before a single path can be simulated. |
| **C** | Risk-neutral densities | $q(k) = g(k)\,\varphi(d_2)/\sqrt{w}$ (Breeden–Litzenberger in log-moneyness): density positivity ⇔ butterfly. Negative density mass is the violation expressed in probability units. |
| **E** | Residual economics | Are (quote − surface) residuals on held-out options economically informative, and does signal quality degrade when the surface itself is arbitrage-dirty (λ=0 counterfactual)? Framed as *economic significance of smoothing residuals*, explicitly **not** as an alpha claim. |
| **D** | VIX replication (bonus) | The 30-day variance-swap rate integrated from each surface vs the published VIX: an **external** daily ground truth for wing quality. Gated on a VIX csv. |

**Inputs** (all optional — sections skip gracefully):
- `option_prices_clean.parquet` (NB01)
- `nb03_surfaces/*.npz` — deep smoother packs with **exact autodiff** fields (w, ∂τw, g); the
  λ=0 pack is the arbitrage-dirty counterfactual for §E
- `nb04_surfaces/*.npz` — operator packs (w only; derivatives by *validated* finite differences
  on the fine export grid — the FD-vs-exact asymmetry is documented and bounded below)
- SVI/SSVI are refit **here** from the quotes (compact NB02/NB03 copies, GJ-certified prior),
  so this notebook is self-contained for the parametric baselines.

**Holdout protocol.** NB05 draws its own 20% per-day holdout with a `crc32(date)+1` seed, applied
*identically to every family* — internal consistency across families is what the comparisons
need. (NB03's holdout uses a per-exdate iteration that is not worth replicating bit-for-bit.)

## 0. Config, loaders, Black machinery

In [1]:
import os, glob, zlib, time
from pathlib import Path

import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize
from scipy.interpolate import RegularGridInterpolator

OUT_DIR   = Path(os.environ.get("THESIS_OUT_DIR", "data/clean"))
REAL_PARQUET = Path(os.environ.get("THESIS_OPT_PARQUET", str(OUT_DIR / "option_prices_clean.parquet")))
NB03_SURF = OUT_DIR / "nb03_surfaces"
NB04_SURF = OUT_DIR / "nb04_surfaces"
VIX_CSV   = Path(os.environ.get("THESIS_VIX_CSV", str(OUT_DIR / "vix_daily.csv")))  # date,vix

SEED           = 0
HOLDOUT_FRAC   = 0.20
VIOL_TOL_MAT   = 1e-3     # same materiality threshold as NB03/NB04
MIN_PTS_SLICE  = 6
V_LOC_FLOOR    = 1e-4     # repaired local-variance bounds (report the repair rate, always)
V_LOC_CAP      = 4.0
MC_PATHS       = 20_000
MC_STEPS       = 120
FLAT_COST_VP   = 0.5      # fallback half-spread (vol points) when quote spreads are unavailable

rng = np.random.default_rng(SEED)
print(f"NB03 packs: {NB03_SURF.exists()} | NB04 packs: {NB04_SURF.exists()} | "
      f"quotes: {REAL_PARQUET.exists()} | VIX csv: {VIX_CSV.exists()}")


def holdout_mask(date, n):
    """NB05's uniform holdout: same rng for every family on a given day."""
    r = np.random.default_rng(zlib.crc32(str(date).encode()) + 1)
    m = r.random(n) < HOLDOUT_FRAC
    if m.all() or (~m).sum() < 20:
        m[:] = False
    return m


# ---------- normalized (undiscounted, forward) Black machinery ----------
from math import erf, sqrt as _sqrt


def _Phi(x):
    return 0.5 * (1.0 + np.vectorize(erf)(np.asarray(x, float) / np.sqrt(2.0)))


def _phi(x):
    return np.exp(-0.5 * np.asarray(x, float) ** 2) / np.sqrt(2 * np.pi)


def black_call(k, w):
    """Normalized undiscounted call C/F with total variance w at log-moneyness k = log(K/F)."""
    w = np.maximum(np.asarray(w, float), 1e-12)
    sw = np.sqrt(w)
    d1 = (-np.asarray(k, float) + w / 2) / sw
    return _Phi(d1) - np.exp(k) * _Phi(d1 - sw)


def black_put(k, w):
    return black_call(k, w) - 1.0 + np.exp(k)      # parity: c - p = 1 - e^k


def black_vega_w(k, w):
    """d(price)/d(sqrt-of-w) i.e. per unit of total-vol; used only for weighting."""
    w = np.maximum(np.asarray(w, float), 1e-12)
    sw = np.sqrt(w)
    return _phi((-np.asarray(k, float) + w / 2) / sw)


def implied_w(price, k, is_call, lo=1e-10, hi=25.0, iters=80):
    """Vectorized bisection for total variance from a normalized price. Monotone in w."""
    price = np.asarray(price, float); k = np.asarray(k, float)
    lo = np.full_like(price, lo); hi = np.full_like(price, hi)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        pm = np.where(is_call, black_call(k, mid), black_put(k, mid))
        too_low = pm < price
        lo = np.where(too_low, mid, lo)
        hi = np.where(too_low, hi, mid)
    return 0.5 * (lo + hi)


# self-test: round trip price -> w
_k = np.array([-0.3, 0.0, 0.2]); _w = np.array([0.04, 0.02, 0.06])
assert np.max(np.abs(implied_w(black_call(_k, _w), _k, np.array([True]*3)) - _w)) < 1e-7
print("Black round-trip self-test OK")

NB03 packs: True | NB04 packs: True | quotes: True | VIX csv: False
Black round-trip self-test OK


## 0b. Surface packs and parametric baselines

A `SurfacePack` exposes `w(k,τ)`, `wt(k,τ)`, `g(k,τ)` by bilinear interpolation on the exported
fine grid. NB03 packs carry **exact autodiff** `wt`/`g`; NB04 packs carry `w` only, so those two
fields are built by central finite differences **on the fine export grid** and the construction
is validated against the analytic SSVI baseline below (max error printed — it must sit far below
`VIOL_TOL_MAT`, otherwise every downstream number would be dominated by differentiation noise).

In [2]:
def durrleman_g(k, w, wk, wkk):
    return (1 - k * wk / (2 * w)) ** 2 - (wk ** 2 / 4) * (1 / w + 0.25) + wkk / 2


def fd_fields(kg, tg, W):
    """wt and g by central differences with COORDINATE-AWARE spacing (np.gradient with the
    coordinate arrays): supports the hybrid (exp U uniform) tau export grid, whose short-end
    density is what keeps the d_tau w reconstruction error ~3e-5 -- the same lesson as the
    collocation chapter: derivative accuracy needs nodes where curvature lives."""
    Wk  = np.gradient(W, kg, axis=1)
    Wkk = np.gradient(Wk, kg, axis=1)
    Wt  = np.gradient(W, tg, axis=0)
    G = durrleman_g(kg[None, :], np.maximum(W, 1e-12), Wk, Wkk)
    return Wt, G


def tau_hybrid(t_lo, t_hi, nt=61):
    """exp U uniform tau grid (NB03's collocation design, reused as the export grid)."""
    te = np.exp(np.linspace(np.log(max(t_lo, 1e-4)), np.log(t_hi), nt))
    tu = np.linspace(t_lo, t_hi, nt)
    return np.unique(np.round(np.concatenate([te, tu]), 10))


class SurfacePack:
    def __init__(self, kg, tg, W, Wt=None, G=None, model="?", tag="", date=""):
        self.k, self.t, self.W = np.asarray(kg, float), np.asarray(tg, float), np.asarray(W, float)
        if Wt is None or G is None:
            Wt, G = fd_fields(self.k, self.t, self.W)
            self.exact = False
        else:
            self.exact = True
        self.Wt, self.G = np.asarray(Wt, float), np.asarray(G, float)
        self.model, self.tag, self.date = model, tag, date
        self._iw  = RegularGridInterpolator((self.t, self.k), self.W,  bounds_error=False, fill_value=None)
        self._iwt = RegularGridInterpolator((self.t, self.k), self.Wt, bounds_error=False, fill_value=None)
        self._ig  = RegularGridInterpolator((self.t, self.k), self.G,  bounds_error=False, fill_value=None)

    def w(self, k, tau):  return np.maximum(self._iw(np.stack([np.clip(tau, self.t[0], self.t[-1]),
                                                               np.clip(k, self.k[0], self.k[-1])], -1)), 1e-12)
    def wt(self, k, tau): return self._iwt(np.stack([np.clip(tau, self.t[0], self.t[-1]),
                                                     np.clip(k, self.k[0], self.k[-1])], -1))
    def g(self, k, tau):  return self._ig(np.stack([np.clip(tau, self.t[0], self.t[-1]),
                                                    np.clip(k, self.k[0], self.k[-1])], -1))
    def iv(self, k, tau): return np.sqrt(self.w(k, tau) / np.maximum(tau, 1e-9))


def load_npz_packs(folder, model_filter=None):
    packs = {}
    for fn in sorted(glob.glob(str(folder / "*.npz"))):
        z = np.load(fn, allow_pickle=True)
        model = str(z["model"]); tag = str(z["tag"]); date = str(z["date"])
        if model_filter and model_filter not in model:
            continue
        key = (model if tag in ("", "lam10") else f"{model}_{tag}", date)
        packs[key] = SurfacePack(z["k"], z["tau"], z["w"],
                                 z["wt"] if "wt" in z.files else None,
                                 z["g"] if "g" in z.files else None,
                                 model=key[0], tag=tag, date=date)
    return packs


packs = {}
if NB03_SURF.exists():
    packs.update(load_npz_packs(NB03_SURF))
if NB04_SURF.exists():
    packs.update(load_npz_packs(NB04_SURF))
DATES = sorted({d for (_, d) in packs})
MODELS = sorted({m for (m, _) in packs})
print(f"loaded {len(packs)} pack(s) | models: {MODELS} | dates: {DATES}")

# ---------- SSVI baseline: GJ-certified prior refit here per day (compact NB03 copy) ----------
GJ_MARGIN = 3.8


def ssvi_w(k, theta, rho, eta, gamma):
    phi = eta * theta ** (-gamma)
    return 0.5 * theta * (1 + rho * phi * k + np.sqrt((phi * k + rho) ** 2 + (1 - rho ** 2)))


def gj_ok(rho, eta, gamma, alpha, beta, t_lo, t_hi):
    th_max = alpha * t_hi ** beta; th_min = alpha * max(t_lo, 1e-6) ** beta
    c1 = eta * th_max ** (1 - gamma) * (1 + abs(rho))
    c2 = eta ** 2 * max(th_max ** (1 - 2 * gamma), th_min ** (1 - 2 * gamma)) * (1 + abs(rho))
    return (0 < gamma < 1) and (c1 < 4) and (c2 <= 4)


def fit_ssvi(kq, tq, wq, wtq):
    taus = np.unique(tq)
    t_lo_, t_hi_ = float(tq.min()), float(tq.max())
    th_t, th_v = [], []
    for t in taus:
        kk_, ww_ = kq[tq == t], wq[tq == t]
        j = int(np.argmin(np.abs(kk_)))
        if abs(float(kk_[j])) <= 0.20 and float(ww_[j]) > 0:
            th_t.append(float(t)); th_v.append(float(ww_[j]))
    if len(th_t) >= 2:
        A = np.vstack([np.ones(len(th_t)), np.log(th_t)]).T
        coef, *_ = np.linalg.lstsq(A, np.log(th_v), rcond=None)
        a0, b0 = float(np.exp(coef[0])), float(np.clip(coef[1], 0.3, 1.5))
    else:
        a0, b0 = float(np.median(wq / tq ** 0.95)), 0.95

    def sse(p):
        rho, eta, gamma, alpha, beta = p
        rho = float(np.clip(rho, -0.95, 0.95)); eta = max(eta, 1e-4)
        gamma = float(np.clip(gamma, 0.05, 0.95)); alpha = max(alpha, 1e-6)
        beta = float(np.clip(beta, 0.3, 1.5))
        tot = sum(float(np.sum(wtq[tq == t] * (ssvi_w(kq[tq == t], alpha * t ** beta,
                                                      rho, eta, gamma) - wq[tq == t]) ** 2))
                  for t in taus)
        if not gj_ok(rho, eta, gamma, alpha, beta, t_lo_, t_hi_):
            tot += 1e3
        return tot

    best = None
    for x0 in [(-0.5, 1.0, 0.3, a0, b0), (-0.7, 0.6, 0.4, a0, b0)]:
        r = minimize(sse, x0, method="Nelder-Mead", options={"maxiter": 1200})
        if best is None or r.fun < best.fun:
            best = r
    rho, eta, gamma, alpha, beta = best.x
    p = dict(rho=float(np.clip(rho, -0.95, 0.95)), eta=float(max(eta, 1e-4)),
             gamma=float(np.clip(gamma, 0.05, 0.95)), alpha=float(max(alpha, 1e-6)),
             beta=float(np.clip(beta, 0.3, 1.5)))
    assert gj_ok(**p, t_lo=t_lo_, t_hi=t_hi_), "SSVI escaped the GJ region"
    return p


def ssvi_pack(pp, k_lo, k_hi, t_lo, t_hi, date):
    """SSVI surface pack with (near-)exact derivatives: central differences with a tiny step on
    the ANALYTIC function -- accurate to ~1e-9, i.e. exact for every purpose here."""
    kg = np.linspace(k_lo, k_hi, 81); tg = np.linspace(t_lo, t_hi, 41)
    KK, TT = np.meshgrid(kg, tg)
    f = lambda K, T: ssvi_w(K, pp["alpha"] * T ** pp["beta"], pp["rho"], pp["eta"], pp["gamma"])
    h = 1e-5
    W  = f(KK, TT)
    Wk = (f(KK + h, TT) - f(KK - h, TT)) / (2 * h)
    Wkk = (f(KK + h, TT) - 2 * W + f(KK - h, TT)) / h ** 2
    Wt = (f(KK, TT + h) - f(KK, TT - h)) / (2 * h)
    G = durrleman_g(KK, np.maximum(W, 1e-12), Wk, Wkk)
    return SurfacePack(kg, tg, W, Wt, G, model="ssvi", date=date)


# ---------- quotes loader (defensive on schema) ----------
def load_day_quotes(date):
    if not REAL_PARQUET.exists():
        return None
    cols = pl.scan_parquet(REAL_PARQUET).collect_schema().names()
    iv_col = "iv_om" if "iv_om" in cols else "impl_volatility"
    want = [c for c in ["date", "exdate", "tau", "k", iv_col, "strike_price", "strike",
                        "best_bid", "best_offer", "forward"] if c in cols]
    df = (pl.scan_parquet(REAL_PARQUET)
            .filter((pl.col("date").cast(pl.Utf8) == str(date)) & pl.col("is_otm"))
            .select(want).drop_nulls([iv_col, "k", "tau"]).collect(engine="streaming"))
    return df.rename({iv_col: "iv"}) if df.height else None


# validate the FD field construction against analytic SSVI (bounds the NB04-pack asymmetry).
# The reference grid mirrors the NB04 v2 EXPORT grid exactly: uniform k (321) x HYBRID tau (61
# per family). NB03 packs ship exact autodiff fields and never enter this code path.
_pp = dict(rho=-0.55, eta=0.9, gamma=0.42, alpha=0.045, beta=0.95)
_kv = np.linspace(-0.5, 0.35, 321)
_tv = tau_hybrid(0.06, 1.5, 61)
_KV, _TV = np.meshgrid(_kv, _tv)
_fw = lambda K, T: ssvi_w(K, _pp["alpha"] * T ** _pp["beta"], _pp["rho"], _pp["eta"], _pp["gamma"])
_h = 1e-5
_Wv = _fw(_KV, _TV)
_Wtv = (_fw(_KV, _TV + _h) - _fw(_KV, _TV - _h)) / (2 * _h)
_Gv = durrleman_g(_KV, np.maximum(_Wv, 1e-12),
                  (_fw(_KV + _h, _TV) - _fw(_KV - _h, _TV)) / (2 * _h),
                  (_fw(_KV + _h, _TV) - 2 * _Wv + _fw(_KV - _h, _TV)) / _h ** 2)
_ref = SurfacePack(_kv, _tv, _Wv, _Wtv, _Gv, model="ref")
_fd = SurfacePack(_ref.k, _ref.t, _ref.W, model="fd_check")   # forces FD reconstruction
_int = (slice(2, -2), slice(2, -2))
err_g = np.max(np.abs(_fd.G[_int] - _ref.G[_int]))
err_wt = np.max(np.abs(_fd.Wt[_int] - _ref.Wt[_int]))
print(f"FD-vs-analytic field validation (interior): |g err| = {err_g:.2e}, "
      f"|wt err| = {err_wt:.2e}  (must be << {VIOL_TOL_MAT})")
assert err_g < VIOL_TOL_MAT / 5 and err_wt < VIOL_TOL_MAT / 5, \
    "FD grid too coarse for material-level auditing; increase export resolution"

loaded 22 pack(s) | models: ['deep', 'deep_lam0', 'deeponet_R1 real-only', 'gno_R3 pretrain+finetune'] | dates: ['2018-01-08', '2025-08-18', '2025-08-19', '2025-08-20', '2025-08-21', '2025-08-22', '2025-08-25', '2025-08-26', '2025-08-27', '2025-08-28', '2025-08-29']
FD-vs-analytic field validation (interior): |g err| = 1.23e-04, |wt err| = 3.38e-05  (must be << 0.001)


## A. Dupire local volatility — the constraints priced

In total-variance/log-moneyness coordinates the Dupire local variance is exactly
$$ \sigma^2_{loc}(k,\tau) \;=\; \frac{\partial_\tau w(k,\tau)}{g(k,\tau)}, $$
where $g$ is the Durrleman density factor used throughout NB02–NB04 (expanding Gatheral's
denominator recovers $g$ term by term). **Numerator = the calendar constraint, denominator = the
butterfly constraint**: a violated surface does not merely trip a flag, it produces a negative or
unbounded local variance that must be *repaired* before one Monte-Carlo path can be drawn. The
repair rate is therefore the headline metric — it is the fraction of the surface on which the
smoother's output is unusable as a pricing model without intervention. This section fills the
gap Chataigner, Crépey & Dixon leave explicitly open (local volatility "from implied
volatilities … in a further paper"), with their §7 protocol: validity, day-to-day stability, and
a Monte-Carlo repricing error.

Two repricing errors are reported:
- **round-trip** (MC with $\sigma_{loc}$ vs the surface's *own* prices): dynamic self-consistency
  of the smoother — an arbitrage-clean surface must reprice itself through its local vol;
- **market** (MC vs held-out quote IVs): the end-to-end economic error.

In [3]:
def local_variance(pack):
    v = pack.Wt / np.where(np.abs(pack.G) < 1e-12, np.nan, pack.G)
    invalid = ~np.isfinite(v) | (v <= 0) | (pack.G <= 0) | (pack.Wt < 0)
    v_rep = np.clip(np.nan_to_num(v, nan=V_LOC_FLOOR), V_LOC_FLOOR, V_LOC_CAP)
    return v, v_rep, invalid


def mc_reprice(pack, k_eval, t_eval, n_paths=MC_PATHS, n_steps=MC_STEPS, seed=SEED):
    """Forward-measure local-vol MC: dX = -1/2 v_loc(X,t) dt + sqrt(v_loc) dW, X0 = 0.
    Prices normalized OTM options at (k_eval, t_eval); returns implied w. Time grid floored at
    the pack's tau_min (below it the surface is uncertified extrapolation; v_loc is held flat)."""
    _, v_rep, _ = local_variance(pack)
    interp = RegularGridInterpolator((pack.t, pack.k), v_rep, bounds_error=False, fill_value=None)
    T = float(np.max(t_eval))
    dt = T / n_steps
    r = np.random.default_rng(seed)
    X = np.zeros(n_paths)
    t_hit = np.sort(np.unique(np.round(np.asarray(t_eval, float), 10)))
    snaps = {}
    t_now, i_hit = 0.0, 0
    for _ in range(n_steps):
        tq = np.clip(t_now, pack.t[0], pack.t[-1])
        v = interp(np.stack([np.full(n_paths, tq), np.clip(X, pack.k[0], pack.k[-1])], -1))
        v = np.clip(v, V_LOC_FLOOR, V_LOC_CAP)
        X = X - 0.5 * v * dt + np.sqrt(v * dt) * r.standard_normal(n_paths)
        t_now += dt
        while i_hit < len(t_hit) and t_now >= t_hit[i_hit] - 1e-12:
            snaps[t_hit[i_hit]] = X.copy()
            i_hit += 1
    w_mc = np.empty(len(k_eval))
    for i, (k, t) in enumerate(zip(k_eval, t_eval)):
        Xt = snaps[min(snaps, key=lambda s: abs(s - t))]
        if k <= 0:
            price = np.mean(np.maximum(np.exp(k) - np.exp(Xt), 0.0))
            w_mc[i] = implied_w(np.array([max(price, 1e-10)]), np.array([k]), np.array([False]))[0]
        else:
            price = np.mean(np.maximum(np.exp(Xt) - np.exp(k), 0.0))
            w_mc[i] = implied_w(np.array([max(price, 1e-10)]), np.array([k]), np.array([True]))[0]
    return w_mc


rows_A = []
for date in DATES:
    q = load_day_quotes(date)
    day_packs = {m: p for (m, d), p in packs.items() if d == date and not m.endswith("lam0")}
    if q is not None:
        kq, tq, ivq = q["k"].to_numpy(), q["tau"].to_numpy(), q["iv"].to_numpy()
        wq = ivq ** 2 * tq
        wtq = 1.0 / (4 * np.maximum(wq, 1e-10) * tq); wtq = wtq / wtq.mean()
        pp = fit_ssvi(kq, tq, wq, wtq)
        day_packs["ssvi"] = ssvi_pack(pp, kq.min(), kq.max(), tq.min(), tq.max(), date)
    for m, pack in sorted(day_packs.items()):
        v, v_rep, invalid = local_variance(pack)
        repair_pct = float(100 * invalid.mean())
        # round-trip + market repricing on a manageable evaluation set
        if q is not None:
            hold = holdout_mask(date, len(kq))
            sel = np.where(hold if hold.any() else np.ones(len(kq), bool))[0]
            sel = sel[np.argsort(np.abs(kq[sel]))][:120]     # densest-information subset
            ke, te, ive = kq[sel], tq[sel], ivq[sel]
            ke = np.clip(ke, pack.k[0] + 1e-6, pack.k[-1] - 1e-6)
            w_mc = mc_reprice(pack, ke, te)
            iv_mc = np.sqrt(w_mc / te)
            iv_surf = pack.iv(ke, te)
            rt = float(np.sqrt(np.mean((iv_mc - iv_surf) ** 2)) * 100)
            mk = float(np.sqrt(np.mean((iv_mc - ive) ** 2)) * 100)
        else:
            rt = mk = None
        rows_A.append(dict(model=m, date=date, repair_pct=repair_pct,
                           min_vloc=float(np.nanmin(v)), max_vloc=float(np.nanmax(v)),
                           mc_roundtrip_rmse_vp=rt, mc_market_rmse_vp=mk,
                           exact_fields=pack.exact))
        print(f"{date} [{m:>14}] repair {repair_pct:5.1f}% of grid | "
              f"v_loc in [{np.nanmin(v):+.4f},{np.nanmax(v):+.4f}] | "
              f"MC round-trip {rt if rt is None else f'{rt:.3f}'} vp | "
              f"MC vs market {mk if mk is None else f'{mk:.3f}'} vp | "
              f"{'exact' if pack.exact else 'FD'} fields")

if rows_A:
    dfA = pl.DataFrame(rows_A)
    dfA.write_parquet(OUT_DIR / "nb05_local_vol.parquet")
    print("written:", OUT_DIR / "nb05_local_vol.parquet")

# day-over-day local-vol stability (needs >= 2 consecutive dates per model)
stab = []
for m in {mm for (mm, _) in packs} | ({"ssvi"} if REAL_PARQUET.exists() else set()):
    ds = [d for (mm, d) in packs if mm == m]
    for d0, d1 in zip(sorted(ds)[:-1], sorted(ds)[1:]):
        p0, p1 = packs[(m, d0)], packs[(m, d1)]
        kg = np.linspace(max(p0.k[0], p1.k[0]), min(p0.k[-1], p1.k[-1]), 41)
        tg = np.linspace(max(p0.t[0], p1.t[0]), min(p0.t[-1], p1.t[-1]), 21)
        KK, TT = np.meshgrid(kg, tg)
        s0 = np.sqrt(np.clip(local_variance(p0)[1], V_LOC_FLOOR, V_LOC_CAP))
        s1 = np.sqrt(np.clip(local_variance(p1)[1], V_LOC_FLOOR, V_LOC_CAP))
        i0 = RegularGridInterpolator((p0.t, p0.k), s0)(np.stack([TT, KK], -1))
        i1 = RegularGridInterpolator((p1.t, p1.k), s1)(np.stack([TT, KK], -1))
        stab.append(dict(model=m, d0=d0, d1=d1,
                         mean_abs_shift=float(np.mean(np.abs(i1 - i0)) * 100)))
if stab:
    print(pl.DataFrame(stab))

# local-vol heatmaps for the latest date
if DATES:
    d = DATES[-1]
    show = {m: p for (m, dd), p in packs.items() if dd == d and not m.endswith("lam0")}
    if show:
        fig = make_subplots(rows=1, cols=len(show),
                            subplot_titles=[f"{m}: sigma_loc (red = repaired)" for m in sorted(show)])
        for j, m in enumerate(sorted(show), start=1):
            p = show[m]
            v, v_rep, invalid = local_variance(p)
            fig.add_trace(go.Heatmap(z=np.sqrt(v_rep), x=p.k, y=p.t, colorscale="Viridis",
                                     showscale=(j == len(show)), colorbar=dict(title="sigma_loc")), 1, j)
            M = np.where(invalid, 1.0, np.nan)
            fig.add_trace(go.Heatmap(z=M, x=p.k, y=p.t, showscale=False,
                                     colorscale=[[0, "rgba(214,39,40,0.85)"], [1, "rgba(214,39,40,0.85)"]]), 1, j)
        fig.update_layout(width=380 * len(show) + 120, height=420,
                          title=f"Dupire local volatility by family — {d} "
                                f"(red cells: v_loc invalid, repaired before simulation)")
        fig.show()

2018-01-08 [          deep] repair   0.0% of grid | v_loc in [+0.0046,+4.5147] | MC round-trip None vp | MC vs market None vp | exact fields


2025-08-18 [deeponet_R1 real-only] repair   1.0% of grid | v_loc in [-232.5744,+72.4817] | MC round-trip None vp | MC vs market None vp | FD fields
2025-08-18 [gno_R3 pretrain+finetune] repair  49.8% of grid | v_loc in [-261.9426,+180.7029] | MC round-trip None vp | MC vs market None vp | FD fields
2025-08-19 [deeponet_R1 real-only] repair   0.9% of grid | v_loc in [-0.0278,+14.8850] | MC round-trip None vp | MC vs market None vp | FD fields
2025-08-19 [gno_R3 pretrain+finetune] repair  49.8% of grid | v_loc in [-675.9780,+36.9018] | MC round-trip None vp | MC vs market None vp | FD fields


2025-08-20 [deeponet_R1 real-only] repair   0.9% of grid | v_loc in [-0.0253,+2.9426] | MC round-trip None vp | MC vs market None vp | FD fields
2025-08-20 [gno_R3 pretrain+finetune] repair  50.0% of grid | v_loc in [-5803.9471,+419.3384] | MC round-trip None vp | MC vs market None vp | FD fields


2025-08-21 [deeponet_R1 real-only] repair   0.8% of grid | v_loc in [-0.0210,+4.2685] | MC round-trip None vp | MC vs market None vp | FD fields
2025-08-21 [gno_R3 pretrain+finetune] repair  50.0% of grid | v_loc in [-238.5071,+439.0427] | MC round-trip None vp | MC vs market None vp | FD fields
2025-08-22 [deeponet_R1 real-only] repair   1.1% of grid | v_loc in [-0.0376,+67.4228] | MC round-trip None vp | MC vs market None vp | FD fields
2025-08-22 [gno_R3 pretrain+finetune] repair  50.0% of grid | v_loc in [-329.6515,+263.0476] | MC round-trip None vp | MC vs market None vp | FD fields


2025-08-25 [deeponet_R1 real-only] repair   1.0% of grid | v_loc in [-0.0328,+36.6131] | MC round-trip None vp | MC vs market None vp | FD fields
2025-08-25 [gno_R3 pretrain+finetune] repair  49.8% of grid | v_loc in [-705.1812,+113.0956] | MC round-trip None vp | MC vs market None vp | FD fields


2025-08-26 [deeponet_R1 real-only] repair   1.0% of grid | v_loc in [-0.0340,+33.1509] | MC round-trip None vp | MC vs market None vp | FD fields
2025-08-26 [gno_R3 pretrain+finetune] repair  49.8% of grid | v_loc in [-1991.3948,+1189.2735] | MC round-trip None vp | MC vs market None vp | FD fields
2025-08-27 [deeponet_R1 real-only] repair   1.0% of grid | v_loc in [-0.0350,+35.4946] | MC round-trip None vp | MC vs market None vp | FD fields
2025-08-27 [gno_R3 pretrain+finetune] repair  50.4% of grid | v_loc in [-60.0783,+422.1091] | MC round-trip None vp | MC vs market None vp | FD fields


2025-08-28 [deeponet_R1 real-only] repair   1.0% of grid | v_loc in [-0.0340,+4.7129] | MC round-trip None vp | MC vs market None vp | FD fields
2025-08-28 [gno_R3 pretrain+finetune] repair  50.0% of grid | v_loc in [-84.5550,+183.4069] | MC round-trip None vp | MC vs market None vp | FD fields


2025-08-29 [deeponet_R1 real-only] repair   1.0% of grid | v_loc in [-0.0310,+3.5340] | MC round-trip None vp | MC vs market None vp | FD fields
2025-08-29 [gno_R3 pretrain+finetune] repair  50.4% of grid | v_loc in [-244.0691,+605.1622] | MC round-trip None vp | MC vs market None vp | FD fields
written: data/clean/nb05_local_vol.parquet
shape: (18, 4)
┌──────────────────────────┬────────────┬────────────┬────────────────┐
│ model                    ┆ d0         ┆ d1         ┆ mean_abs_shift │
│ ---                      ┆ ---        ┆ ---        ┆ ---            │
│ str                      ┆ str        ┆ str        ┆ f64            │
╞══════════════════════════╪════════════╪════════════╪════════════════╡
│ gno_R3 pretrain+finetune ┆ 2025-08-18 ┆ 2025-08-19 ┆ 8.339637       │
│ gno_R3 pretrain+finetune ┆ 2025-08-19 ┆ 2025-08-20 ┆ 7.371514       │
│ gno_R3 pretrain+finetune ┆ 2025-08-20 ┆ 2025-08-21 ┆ 7.095737       │
│ gno_R3 pretrain+finetune ┆ 2025-08-21 ┆ 2025-08-22 ┆ 6.967389      

## C. Risk-neutral densities (Breeden–Litzenberger) — butterfly in probability units

In log-moneyness the risk-neutral density of $X_T=\log(S_T/F_T)$ is
$q(k) = g(k)\,\varphi(d_2)/\sqrt{w}$ with $d_2 = -k/\sqrt{w} - \sqrt{w}/2$: **positivity ⇔
butterfly**, so negative mass is the violation stated in the units a risk manager reads. Also
checked: $\int q\,dk \approx 1$ and the martingale identity $\int e^k q\,dk \approx 1$
(deviations on the *truncated* export domain are reported as truncation, not error).

In [4]:
def bl_density(pack, tau):
    kg = pack.k
    w = pack.w(kg, np.full_like(kg, tau))
    g = pack.g(kg, np.full_like(kg, tau))
    sw = np.sqrt(w)
    d2 = -kg / sw - sw / 2
    q = g * _phi(d2) / sw
    return kg, q


rows_C = []
for date in DATES:
    day_packs = {m: p for (m, d), p in packs.items() if d == date and not m.endswith("lam0")}
    for m, pack in sorted(day_packs.items()):
        for tau in np.quantile(pack.t, [0.15, 0.5, 0.9]):
            kg, q = bl_density(pack, float(tau))
            dk = kg[1] - kg[0]
            rows_C.append(dict(model=m, date=date, tau=float(tau),
                               neg_mass_pct=float(100 * np.sum(np.abs(q[q < 0])) * dk),
                               total_mass=float(np.sum(q) * dk),
                               martingale=float(np.sum(np.exp(kg) * q) * dk),
                               min_q=float(q.min())))
if rows_C:
    dfC = pl.DataFrame(rows_C)
    dfC.write_parquet(OUT_DIR / "nb05_densities.parquet")
    print(dfC)

if DATES:
    d = DATES[-1]
    fig = go.Figure()
    for m, pack in sorted({m: p for (m, dd), p in packs.items()
                           if dd == d and not m.endswith("lam0")}.items()):
        tau = float(np.quantile(pack.t, 0.5))
        kg, q = bl_density(pack, tau)
        fig.add_trace(go.Scatter(x=kg, y=q, name=f"{m} (tau={tau:.2f})"))
    fig.add_hline(y=0, line_dash="dot")
    fig.update_layout(width=880, height=420, xaxis_title="log-moneyness k",
                      yaxis_title="risk-neutral density q(k)",
                      title=f"Breeden–Litzenberger densities — {d} "
                            f"(any excursion below 0 is butterfly arbitrage)")
    fig.show()

shape: (63, 7)
┌────────────────────┬────────────┬──────────┬──────────────┬────────────┬────────────┬────────────┐
│ model              ┆ date       ┆ tau      ┆ neg_mass_pct ┆ total_mass ┆ martingale ┆ min_q      │
│ ---                ┆ ---        ┆ ---      ┆ ---          ┆ ---        ┆ ---        ┆ ---        │
│ str                ┆ str        ┆ f64      ┆ f64          ┆ f64        ┆ f64        ┆ f64        │
╞════════════════════╪════════════╪══════════╪══════════════╪════════════╪════════════╪════════════╡
│ deep               ┆ 2018-01-08 ┆ 0.310822 ┆ 0.0          ┆ 1.009862   ┆ 1.010189   ┆ 3.4518e-7  │
│ deep               ┆ 2018-01-08 ┆ 0.984932 ┆ 0.0          ┆ 0.992897   ┆ 0.990452   ┆ 0.0004     │
│ deep               ┆ 2018-01-08 ┆ 1.755342 ┆ 0.0          ┆ 0.979939   ┆ 0.97375    ┆ 0.002122   │
│ deeponet_R1        ┆ 2025-08-18 ┆ 0.133574 ┆ 0.0          ┆ 0.999157   ┆ 0.999512   ┆ 4.4654e-21 │
│ real-only          ┆            ┆          ┆              ┆            ┆  

## E. Economic significance of smoothing residuals — tightly framed

**Claim under test** (and the only one): residuals $r_i = \text{IV}^{mkt}_i - \text{IV}^{model}_i$
on *held-out* quotes contain economically meaningful information, and that information degrades
when the surface itself is arbitrage-dirty. **This is not an alpha claim**: no slippage model, no
capacity, no borrowing costs — the deliverable is an information coefficient and a cost-aware
decile spread, i.e. *economic significance*, plus the thesis link (clean λ=10 vs dirty λ=0
counterfactual on the same day).

Protocol per (day *t*, family): residuals on NB05's held-out quotes; match each option to day
*t+1* by `(exdate, strike)`; IC = Spearman(r_t, −ΔIV_{t→t+1}); decile portfolio long the cheapest
decile / short the richest, vega-normalized, 1-day hold; costs = per-option IV half-spread from
bid/ask when available (`(ask−bid)/(2·F·vega)`), else a flat `FLAT_COST_VP` clearly flagged.
Statistical power arrives with the full-run exports (`EXPORT_ALL=True` in NB03); with the staging
packs this section is a *pipeline validation*, and says so.

In [5]:
def day_pair(date):
    """This date's quotes and the next trading day's, matched on (exdate, strike)."""
    if not REAL_PARQUET.exists():
        return None, None
    all_dates = (pl.scan_parquet(REAL_PARQUET).select("date").unique()
                   .collect(engine="streaming")["date"].sort().to_list())
    ds = [str(x) for x in all_dates]
    if str(date) not in ds or ds.index(str(date)) + 1 >= len(ds):
        return None, None
    return load_day_quotes(date), load_day_quotes(all_dates[ds.index(str(date)) + 1])


rows_E = []
for date in DATES:
    q0, q1 = day_pair(date)
    if q0 is None or q1 is None:
        continue
    strike_col = next((c for c in ["strike_price", "strike"] if c in q0.columns), None)
    if strike_col is None or "exdate" not in q0.columns:
        print(f"[E] {date}: no (exdate, strike) columns — cannot match across days; skipped.")
        continue
    kq, tq, ivq = q0["k"].to_numpy(), q0["tau"].to_numpy(), q0["iv"].to_numpy()
    hold = holdout_mask(date, len(kq))
    if not hold.any():
        continue
    # next-day IV per (exdate, strike)
    nxt = {(str(e), float(s)): float(v) for e, s, v in
           zip(q1["exdate"].to_list(), q1[strike_col].to_list(), q1["iv"].to_list())}
    ex0, st0 = q0["exdate"].to_list(), q0[strike_col].to_list()

    day_models = {m: p for (m, d), p in packs.items() if d == date}
    wq = ivq ** 2 * tq
    wtq = 1.0 / (4 * np.maximum(wq, 1e-10) * tq); wtq = wtq / wtq.mean()
    if "ssvi" not in day_models:
        day_models["ssvi"] = ssvi_pack(fit_ssvi(kq, tq, wq, wtq),
                                       kq.min(), kq.max(), tq.min(), tq.max(), date)

    # per-option IV half-spread cost, if the schema allows it
    if {"best_bid", "best_offer", "forward"} <= set(q0.columns):
        F = q0["forward"].to_numpy()
        vega = np.maximum(F * _phi((-kq + wq / 2) / np.sqrt(wq)) * np.sqrt(tq), 1e-8)
        cost_iv = (q0["best_offer"].to_numpy() - q0["best_bid"].to_numpy()) / (2 * vega)
        cost_note = "quote half-spreads"
    else:
        cost_iv = np.full(len(kq), FLAT_COST_VP / 100)
        cost_note = f"FLAT {FLAT_COST_VP} vp fallback"

    for m, pack in sorted(day_models.items()):
        idx = np.where(hold)[0]
        kk = np.clip(kq[idx], pack.k[0] + 1e-6, pack.k[-1] - 1e-6)
        r = ivq[idx] - pack.iv(kk, tq[idx])                    # + = quote rich vs surface
        div, keep = [], []
        for j, i in enumerate(idx):
            key = (str(ex0[i]), float(st0[i]))
            if key in nxt:
                div.append(nxt[key] - ivq[i]); keep.append(j)
        if len(keep) < 30:
            continue
        r_, div_, c_ = r[keep], np.array(div), cost_iv[idx][keep]
        # IC: does richness today predict a fall tomorrow?
        from scipy.stats import spearmanr
        ic = float(spearmanr(r_, -div_).statistic)
        # decile spread, vega-normalized units of IV, net of entry costs
        qlo, qhi = np.quantile(r_, [0.1, 0.9])
        long_, short_ = r_ <= qlo, r_ >= qhi                    # long cheap, short rich
        gross = float(np.mean(div_[long_]) - np.mean(div_[short_])) * 100
        costs = float(np.mean(c_[long_]) + np.mean(c_[short_])) * 100
        rows_E.append(dict(model=m, date=date, n_matched=len(keep), ic=ic,
                           decile_spread_gross_vp=gross, entry_costs_vp=costs,
                           decile_spread_net_vp=gross - costs, cost_basis=cost_note))
        print(f"{date} [{m:>14}] n={len(keep):>4} | IC {ic:+.3f} | decile spread "
              f"{gross:+.3f} vp gross / {gross - costs:+.3f} net ({cost_note})")

if rows_E:
    dfE = pl.DataFrame(rows_E)
    dfE.write_parquet(OUT_DIR / "nb05_residual_economics.parquet")
    # the thesis link: clean vs dirty smoother on the same day
    if any(m.endswith("lam0") for m, _ in packs):
        cmp_ = dfE.filter(pl.col("model").str.contains("deep")).sort(["date", "model"])
        print("\nCLEAN (lam=10) vs DIRTY (lam=0) signal quality — same day, same quotes:")
        print(cmp_.select(["date", "model", "ic", "decile_spread_net_vp"]))
    n_days = dfE["date"].n_unique()
    if n_days < 20:
        print(f"\n[power] Only {n_days} day(s) available: this is a PIPELINE VALIDATION. "
              f"Statistical claims require the full-run exports (EXPORT_ALL=True in NB03).")

## D. VIX replication (bonus, gated) — an external ground truth for the wings

The 30-day variance-swap rate from each surface, via the log-strip in forward log-moneyness:
$$ \sigma_{VS}^2(T)\,T \;=\; 2\int \tilde q(k)\,e^{-k}\,dk, \qquad
   \tilde q(k) = \text{normalized OTM Black price at } (k, w(k,T)), $$
with total variance interpolated linearly in $\tau$ to $T=30/365$, compared to the published VIX.
The integral runs over the pack's (truncated) k-range: the truncation deficit is itself
informative — it is largest exactly when the model's wings are poorest. Self-tested against flat
Black-Scholes below. Provide `vix_daily.csv` with columns `date,vix` to activate.

In [6]:
def varswap_rate(pack, T=30 / 365, nk=400):
    t0 = float(np.clip(T, pack.t[0], pack.t[-1]))
    kg = np.linspace(pack.k[0] + 1e-4, pack.k[-1] - 1e-4, nk)
    w = pack.w(kg, np.full_like(kg, t0)) * (T / t0)     # linear-in-tau total variance scaling
    otm = np.where(kg <= 0, black_put(kg, w), black_call(kg, w))
    _trapz = getattr(np, "trapezoid", getattr(np, "trapz", None))
    integ = 2 * _trapz(otm * np.exp(-kg), kg)
    return float(np.sqrt(max(integ, 1e-12) / T))


# self-test on flat BS at sigma = 0.2 over a wide strip
_flat = SurfacePack(np.linspace(-2.0, 2.0, 201), np.linspace(0.02, 1.0, 21),
                    0.04 * np.tile(np.linspace(0.02, 1.0, 21)[:, None], (1, 201)), model="flat")
_vs = varswap_rate(_flat)
print(f"VIX machinery self-test: flat sigma=0.200 -> varswap {_vs:.4f} (truncation-limited)")
assert abs(_vs - 0.20) < 5e-3

if VIX_CSV.exists() and DATES:
    vix = pl.read_csv(VIX_CSV).with_columns(pl.col("date").cast(pl.Utf8))
    rows_D = []
    for date in DATES:
        row = vix.filter(pl.col("date") == str(date))
        if not row.height:
            continue
        v_mkt = float(row["vix"][0]) / 100
        for m, pack in sorted({m: p for (m, d), p in packs.items()
                               if d == date and not m.endswith("lam0")}.items()):
            rows_D.append(dict(model=m, date=date, model_vs=varswap_rate(pack),
                               vix=v_mkt, err_volpts=(varswap_rate(pack) - v_mkt) * 100,
                               k_range=float(pack.k[-1] - pack.k[0])))
    if rows_D:
        dfD = pl.DataFrame(rows_D)
        dfD.write_parquet(OUT_DIR / "nb05_vix.parquet")
        print(dfD)
else:
    print(f"[D] gated: place a date,vix csv at {VIX_CSV} to activate.")

VIX machinery self-test: flat sigma=0.200 -> varswap 0.1997 (truncation-limited)
[D] gated: place a date,vix csv at data/clean/vix_daily.csv to activate.


## Summary

This notebook converts the thesis's audit metrics into pricing objects, one per section:

- **A (local vol)**: $\sigma^2_{loc} = \partial_\tau w / g$ — the *repair rate* is the fraction of
  the surface unusable as a pricing model without intervention, and the MC **round-trip error**
  measures whether a smoother is dynamically consistent with itself. NB03 packs use exact
  autodiff fields; NB04/FD fields are validated to << the materiality threshold.
- **C (densities)**: butterfly violations restated as negative probability mass; total-mass and
  martingale checks with truncation reported separately.
- **E (residual economics)**: IC + cost-aware decile spread on held-out quotes, uniform holdout
  across families, with the λ=0 counterfactual quantifying what arbitrage-dirtiness costs in
  signal quality. Explicitly *not* an alpha claim; power comes with the full-run exports.
- **D (VIX)**: external ground truth for the wings, gated on data availability, self-tested.

**Production checklist for the full run:** (1) NB03 with `LIMIT_DATES=None`, `EXPORT_ALL=True`
(parallelize by year); (2) NB04 on GPU with raised epochs, `NB04_EXPORT_DAYS` to cover the NB03
dates; (3) re-run this notebook — every section scales with whatever packs exist.